# End-to-End Fine-Tuning Loop: Evals → Baseline → Fine-Tune → Compare → Analyze

**Goal:** walk through the complete, repeatable loop that practitioners actually follow when improving a model with fine-tuning — starting and ending with evaluation.

---

## Why Start with Evals?

Most fine-tuning tutorials start with the training step. In practice the first thing you build is the **evaluation harness**, because without it you cannot answer the only question that matters:

> Did fine-tuning actually help?

## The Full Loop

```text
┌──────────────────────────────────────────────────────────┐
│ 1. DEFINE EVALS         What does "better" mean?         │
│    ↓                                                     │
│ 2. BASELINE PROMPTING   How good is the model already?   │
│    ↓                                                     │
│ 3. BUILD DATASET        Curate examples for SFT          │
│    ↓                                                     │
│ 4. FINE-TUNE            LoRA adapter on supervised data   │
│    ↓                                                     │
│ 5. COMPARE              Same evals on baseline vs FT     │
│    ↓                                                     │
│ 6. ERROR ANALYSIS       Where did FT help? Where not?    │
│    ↓                                                     │
│ 7. ITERATE              Fix data → retrain → re-eval     │
└──────────────────────────────────────────────────────────┘
```

## The Task: Ticket Priority Classifier

We classify support tickets into four priority levels:

| Priority | Meaning | Example |
|---|---|---|
| `P0-critical` | Service down, data loss, security breach | "Production database is unreachable" |
| `P1-high` | Major feature broken, workaround exists | "Exports fail for files > 50 MB" |
| `P2-medium` | Minor issue, does not block work | "Dashboard chart legend overlaps on mobile" |
| `P3-low` | Cosmetic, enhancement, question | "Can the logo be a bit larger?" |

This is a realistic narrow task where prompting gets you partway, but fine-tuning captures the nuanced priority rules specific to your organization.


## 1. Environment Setup

In [ ]:
!pip install -q pandas numpy scikit-learn datasets transformers peft trl accelerate seaborn matplotlib

In [ ]:
import json
import random
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = ARTIFACT_DIR / "priority-lora"

RUN_BASELINE = False
RUN_TRAINING = False
RUN_FT_EVAL = False

CLASSES = ["P0-critical", "P1-high", "P2-medium", "P3-low"]
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for c, i in label2id.items()}

print(f"Project dir:  {PROJECT_DIR}")
print(f"Base model:   {BASE_MODEL}")
print(f"Classes:      {CLASSES}")
print(f"Baseline run: {RUN_BASELINE}")
print(f"Training:     {RUN_TRAINING}")
print(f"FT eval:      {RUN_FT_EVAL}")

## 2. Step 1 — Define Evals First

### Why Evals Before Anything Else

If you fine-tune first and scramble to measure afterwards, you risk:
- cherry-picking metrics that make the result look good
- missing regressions on slices you forgot to test
- comparing apples to oranges if the evaluation setup differs between runs

### What We Measure

| Metric | Why |
|---|---|
| **Overall accuracy** | Quick sanity check |
| **Macro F1** | Balanced measure across all classes regardless of frequency |
| **Per-class precision & recall** | Reveals which priorities the model gets wrong |
| **Confusion matrix** | Shows error direction — "Is P1 confused with P2 or with P0?" |
| **Severity confusion rate** | Custom: how often does the model confuse adjacent vs. distant priorities? |
| **Parse success rate** | Does the model output a valid priority label at all? |

### The Evaluation Contract

We fix all of these choices **before** running any model. The same eval function scores the baseline prompt approach and the fine-tuned model identically.


In [ ]:
def parse_priority(text):
    '''Extract a valid priority label from model output.'''
    text_lower = text.strip().lower()
    for cls in CLASSES:
        if cls.lower() in text_lower:
            return cls
    # Fallback: try just the P-level
    for cls in CLASSES:
        tag = cls.split("-")[0].lower()
        if tag in text_lower:
            return cls
    return None


def severity_distance(true_label, pred_label):
    '''How many priority levels apart are the true and predicted labels?
    P0-P1 = 1 step, P0-P3 = 3 steps.'''
    return abs(label2id[true_label] - label2id[pred_label])


def evaluate(true_labels, raw_outputs, run_name="model"):
    '''Full evaluation suite. Returns a structured report dict.'''
    parsed = [parse_priority(o) for o in raw_outputs]
    parse_rate = sum(1 for p in parsed if p is not None) / len(parsed)

    # Replace unparseable with a sentinel for metrics
    preds_for_metrics = [p if p is not None else "UNPARSED" for p in parsed]
    all_labels = CLASSES + ["UNPARSED"]

    acc = accuracy_score(true_labels, preds_for_metrics)
    macro_f1 = f1_score(
        true_labels, preds_for_metrics, labels=CLASSES, average="macro", zero_division=0
    )

    # Per-class report
    cls_report = classification_report(
        true_labels, preds_for_metrics,
        labels=CLASSES, target_names=CLASSES, digits=3, zero_division=0, output_dict=True,
    )

    # Severity confusion (only on parseable predictions)
    distances = []
    for t, p in zip(true_labels, parsed):
        if p is not None:
            distances.append(severity_distance(t, p))
    mean_sev_dist = np.mean(distances) if distances else float("nan")
    severe_errors = sum(1 for d in distances if d >= 2)
    severe_rate = severe_errors / len(distances) if distances else float("nan")

    report = {
        "run": run_name,
        "accuracy": round(acc, 4),
        "macro_f1": round(macro_f1, 4),
        "parse_rate": round(parse_rate, 4),
        "mean_severity_distance": round(mean_sev_dist, 4),
        "severe_error_rate": round(severe_rate, 4),
        "per_class": {c: {k: round(v, 4) for k, v in cls_report[c].items()} for c in CLASSES},
    }
    return report, preds_for_metrics


print("Evaluation suite defined.")
print("Metrics: accuracy, macro_f1, parse_rate, severity_distance, severe_error_rate, per-class P/R/F1")

In [ ]:
def print_report(report):
    print(f"\n{'=' * 65}")
    print(f"  EVALUATION: {report['run']}")
    print(f"{'=' * 65}")
    print(f"  Accuracy:              {report['accuracy']:.1%}")
    print(f"  Macro F1:              {report['macro_f1']:.1%}")
    print(f"  Parse success rate:    {report['parse_rate']:.1%}")
    print(f"  Mean severity distance:{report['mean_severity_distance']:.3f}")
    print(f"  Severe error rate:     {report['severe_error_rate']:.1%}")
    print(f"\n  Per-class breakdown:")
    print(f"  {'Class':<14} {'Prec':>7} {'Recall':>7} {'F1':>7} {'Support':>8}")
    print(f"  {'-'*44}")
    for cls in CLASSES:
        c = report["per_class"][cls]
        print(f"  {cls:<14} {c['precision']:>6.1%} {c['recall']:>6.1%} {c['f1-score']:>6.1%} {c['support']:>8.0f}")
    print()


def plot_confusion(true_labels, pred_labels, title="Confusion Matrix"):
    cm = confusion_matrix(true_labels, pred_labels, labels=CLASSES + ["UNPARSED"])
    # Drop UNPARSED row (we only care about prediction distribution)
    cm_display = cm[:len(CLASSES), :len(CLASSES)]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    sns.heatmap(cm_display, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    axes[0].set_title(f"{title} (counts)")
    cm_norm = cm_display.astype(float) / cm_display.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1])
    axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
    axes[1].set_title(f"{title} (recall-norm)")
    plt.tight_layout()
    plt.show()


print("Reporting and plotting helpers defined.")

## 3. Dataset

### Priority Classification Rules (Organization-Specific)

These rules are what makes fine-tuning worthwhile. A generic LLM does not know your organization's priority definitions:

| Trigger | Priority |
|---|---|
| Service fully down, data loss, security breach | P0-critical |
| Major feature broken but workaround exists, performance > 10× slower | P1-high |
| Minor bug, UI issue, non-blocking, partial feature gap | P2-medium |
| Cosmetic, enhancement request, documentation question | P3-low |

### Split Strategy

- **eval_set**: 20 tickets with known ground-truth labels — used for baseline AND fine-tuned evaluation
- **train_set**: 40 tickets — used only for SFT training
- Same eval set for both runs ensures a fair comparison


In [ ]:
tickets = [
    # ── P0-critical ──
    {"text": "Production database is completely unreachable. All API endpoints returning 503.", "label": "P0-critical"},
    {"text": "Customer data was exposed through an unauthenticated endpoint.", "label": "P0-critical"},
    {"text": "Payment processing is down. No orders can be completed.", "label": "P0-critical"},
    {"text": "Authentication service crashed. No user can log in.", "label": "P0-critical"},
    {"text": "Data corruption detected in the billing table after last migration.", "label": "P0-critical"},
    {"text": "SSL certificate expired on the main domain. Site shows security warning.", "label": "P0-critical"},
    {"text": "Entire Kubernetes cluster is unresponsive. All services affected.", "label": "P0-critical"},
    {"text": "Ransomware alert triggered on the staging server.", "label": "P0-critical"},
    {"text": "Primary DNS failing. Customers cannot reach any of our services.", "label": "P0-critical"},
    {"text": "Webhook delivery system is down. No events reaching partner integrations.", "label": "P0-critical"},
    {"text": "Key management service offline. Cannot decrypt any stored secrets.", "label": "P0-critical"},
    {"text": "Load balancer health checks failing. 50% of traffic getting 502 errors.", "label": "P0-critical"},
    {"text": "CDN purge stuck. Users seeing stale content with wrong pricing.", "label": "P0-critical"},
    {"text": "Database replication lag exceeded 30 minutes. Risk of data loss on failover.", "label": "P0-critical"},
    {"text": "CI/CD pipeline deploying to production without approval gates after config change.", "label": "P0-critical"},

    # ── P1-high ──
    {"text": "CSV export fails for files larger than 50 MB. Users can export smaller files as a workaround.", "label": "P1-high"},
    {"text": "Search results take 45 seconds to load. Was under 2 seconds last week.", "label": "P1-high"},
    {"text": "Email notifications are delayed by 3-4 hours. They do arrive eventually.", "label": "P1-high"},
    {"text": "SSO login broken for Google accounts. Password login still works.", "label": "P1-high"},
    {"text": "Dashboard analytics showing data from 2 days ago instead of real-time.", "label": "P1-high"},
    {"text": "API rate limiter is too aggressive. Legitimate users hitting 429 errors frequently.", "label": "P1-high"},
    {"text": "Mobile app crashes on launch for Android 14 users. iOS works fine.", "label": "P1-high"},
    {"text": "Bulk import feature timing out for datasets above 10k rows. Manual entry works.", "label": "P1-high"},
    {"text": "Webhook retry logic is broken. Failed deliveries are not being retried.", "label": "P1-high"},
    {"text": "PDF report generation produces blank pages for charts. Tables render correctly.", "label": "P1-high"},
    {"text": "Two-factor authentication codes arriving 5 minutes late. Users can retry.", "label": "P1-high"},
    {"text": "Memory leak in the worker process. Requires restart every 8 hours.", "label": "P1-high"},
    {"text": "Permissions API returning stale cache. Users see old roles until cache expires.", "label": "P1-high"},
    {"text": "Scheduled reports are not being sent. Manual report generation still works.", "label": "P1-high"},
    {"text": "File upload progress bar stuck at 99% though upload completes. Confuses users.", "label": "P1-high"},

    # ── P2-medium ──
    {"text": "Chart legend overlaps with the title on screens narrower than 1024px.", "label": "P2-medium"},
    {"text": "Dropdown menu flickers when hovering quickly between items.", "label": "P2-medium"},
    {"text": "Timezone display defaults to UTC instead of the user's local timezone.", "label": "P2-medium"},
    {"text": "The search bar does not clear after navigating to a new page.", "label": "P2-medium"},
    {"text": "Email templates have inconsistent font sizing between sections.", "label": "P2-medium"},
    {"text": "Cursor jumps to end of input field when editing a saved filter name.", "label": "P2-medium"},
    {"text": "Table sorting resets when switching tabs and coming back.", "label": "P2-medium"},
    {"text": "Error message says 'something went wrong' without any detail.", "label": "P2-medium"},
    {"text": "Profile page shows 'member since' date in US format, but user is in EU.", "label": "P2-medium"},
    {"text": "Copy-to-clipboard button does not show a confirmation toast.", "label": "P2-medium"},
    {"text": "API docs example uses deprecated v1 endpoint instead of v2.", "label": "P2-medium"},
    {"text": "Color contrast on the secondary button fails WCAG AA for small text.", "label": "P2-medium"},
    {"text": "Cannot re-order dashboard widgets after saving a custom layout.", "label": "P2-medium"},
    {"text": "Form validation fires before the user finishes typing.", "label": "P2-medium"},
    {"text": "Notification badge count shows unread count from all workspaces, not just the active one.", "label": "P2-medium"},

    # ── P3-low ──
    {"text": "Can the logo on the login page be a bit larger?", "label": "P3-low"},
    {"text": "Would be nice to have keyboard shortcuts for common actions.", "label": "P3-low"},
    {"text": "The empty state illustration feels outdated.", "label": "P3-low"},
    {"text": "Can we add a dark mode option?", "label": "P3-low"},
    {"text": "It would be helpful to see a word count in the text editor.", "label": "P3-low"},
    {"text": "Suggestion: add a 'recently viewed' section to the homepage.", "label": "P3-low"},
    {"text": "The loading spinner could use a smoother animation.", "label": "P3-low"},
    {"text": "Where can I find documentation on the batch API?", "label": "P3-low"},
    {"text": "Can you add an option to collapse the sidebar by default?", "label": "P3-low"},
    {"text": "Feature idea: let users star or bookmark projects.", "label": "P3-low"},
    {"text": "The favicon is blurry on high-DPI screens.", "label": "P3-low"},
    {"text": "Is there a way to export my data as JSON instead of only CSV?", "label": "P3-low"},
    {"text": "Would love to see usage analytics on my own API keys.", "label": "P3-low"},
    {"text": "The onboarding walkthrough could mention the CLI tool.", "label": "P3-low"},
    {"text": "Suggestion: let users customize the default date range on dashboards.", "label": "P3-low"},
]

df = pd.DataFrame(tickets)
print(f"Total tickets: {len(df)}")
print(f"\nClass distribution:")
print(df["label"].value_counts().to_string())

In [ ]:
# Split: stratified eval set (20) + train set (40)
train_df, eval_df = train_test_split(
    df, test_size=20, random_state=SEED, stratify=df["label"]
)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)

print(f"Train: {len(train_df)}")
print(f"Eval:  {len(eval_df)}")
print(f"\nEval class counts: {dict(Counter(eval_df['label']))}")
print(f"Train class counts: {dict(Counter(train_df['label']))}")

## 4. Step 2 — Baseline Prompting

### Why Baseline First

Before investing in dataset curation and training compute, measure how far prompting alone gets you. If the base model already scores 95%, fine-tuning may not be worth it.

### Prompt Design

We give the model:
- the priority definitions
- the ticket text
- an instruction to output exactly one priority label

This is the best we can do without training data — a well-crafted zero-shot prompt.


In [ ]:
CLASSIFICATION_SYSTEM_PROMPT = (
    "You are a support ticket priority classifier.\n\n"
    "Classify each ticket into exactly one priority level:\n"
    "- P0-critical: service fully down, data loss, security breach\n"
    "- P1-high: major feature broken but workaround exists, severe performance degradation\n"
    "- P2-medium: minor bug, UI issue, non-blocking defect\n"
    "- P3-low: cosmetic issue, enhancement request, documentation question\n\n"
    "Reply with ONLY the priority label (e.g. P2-medium). No explanation."
)


def build_classify_messages(ticket_text):
    return [
        {"role": "system", "content": CLASSIFICATION_SYSTEM_PROMPT},
        {"role": "user", "content": ticket_text},
    ]


print("Classification prompt:")
print(CLASSIFICATION_SYSTEM_PROMPT)
print("\nExample messages:")
print(json.dumps(build_classify_messages("Dashboard crashes when I filter by date range."), indent=2))

In [ ]:
if RUN_BASELINE:
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline as hf_pipeline
    import torch

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    generator = hf_pipeline("text-generation", model=model, tokenizer=tokenizer)

    baseline_outputs = []
    for _, row in eval_df.iterrows():
        messages = build_classify_messages(row["text"])
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        result = generator(prompt, max_new_tokens=20, do_sample=False)
        output = result[0]["generated_text"][len(prompt):].strip()
        baseline_outputs.append(output)

    baseline_report, baseline_preds = evaluate(
        eval_df["label"].tolist(), baseline_outputs, run_name="baseline_prompt"
    )
    print_report(baseline_report)
    plot_confusion(eval_df["label"].tolist(), baseline_preds, title="Baseline Prompt")
else:
    # Simulate plausible baseline results: prompting gets ~65% on nuanced priority rules
    rng = np.random.default_rng(SEED)
    sim_baseline_outputs = []
    for _, row in eval_df.iterrows():
        true = row["label"]
        if rng.random() < 0.65:
            sim_baseline_outputs.append(true)
        else:
            # Common prompt errors: confuse adjacent priorities
            neighbors = {
                "P0-critical": ["P1-high"],
                "P1-high": ["P0-critical", "P2-medium"],
                "P2-medium": ["P1-high", "P3-low"],
                "P3-low": ["P2-medium"],
            }
            sim_baseline_outputs.append(rng.choice(neighbors[true]))

    baseline_report, baseline_preds = evaluate(
        eval_df["label"].tolist(), sim_baseline_outputs, run_name="baseline_prompt"
    )
    print_report(baseline_report)
    print("(Simulated results. Set RUN_BASELINE = True for real evaluation.)")

In [ ]:
plot_confusion(eval_df["label"].tolist(), baseline_preds, title="Baseline Prompt")

## 5. Step 3 — Build the Fine-Tuning Dataset

### Key Principle: Same Prompt Template

The training data uses the **exact same system prompt and message structure** as the baseline evaluation above. This ensures the model learns to improve on the same interface — not a different one.

### What the Training Examples Teach

Each example shows the model:
- the ticket text (user message)
- the correct priority label (assistant response)
- implicitly, the boundary between adjacent priority levels

The fine-tuning data encodes your **organization's specific priority rules**, which the base model could not learn from a prompt alone.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def to_chat_record(row):
    messages = [
        {"role": "system", "content": CLASSIFICATION_SYSTEM_PROMPT},
        {"role": "user", "content": row["text"]},
        {"role": "assistant", "content": row["label"]},
    ]
    return {
        "messages": messages,
        "text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False),
    }


train_records = [to_chat_record(row) for _, row in train_df.iterrows()]
train_dataset = Dataset.from_list(train_records)

# Token length stats
train_lengths = [len(tokenizer(r["text"]).input_ids) for r in train_records]
print(f"Training examples: {len(train_records)}")
print(f"Token lengths:     mean={np.mean(train_lengths):.0f}  max={max(train_lengths)}  min={min(train_lengths)}")

# Save JSONL
jsonl_path = ARTIFACT_DIR / "priority_sft.jsonl"
with jsonl_path.open("w", encoding="utf-8") as f:
    for rec in train_records:
        f.write(json.dumps({"messages": rec["messages"]}, ensure_ascii=False) + "\n")
print(f"Saved: {jsonl_path}")

## 6. Step 4 — Fine-Tune with LoRA

### Why LoRA for This Task

Priority classification is a **narrow behavioral adjustment**: the model already understands language, support tickets, and priority concepts. LoRA changes only a small set of weights to specialize the model's judgment on priority boundaries.

### Hyperparameter Choices

| Parameter | Value | Rationale |
|---|---|---|
| LoRA rank | 16 | Enough expressivity for classification boundary learning |
| Epochs | 3 | Small dataset — more epochs risks overfitting the training examples |
| Learning rate | 1e-4 | Standard LoRA range for instruction-tuned models |
| Max seq length | auto | Fits the longest training example |


In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer

MAX_SEQ_LENGTH = max(train_lengths) + 32

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    report_to="none",
    seed=SEED,
)

print(f"LoRA rank:     {peft_config.r}")
print(f"Max seq len:   {MAX_SEQ_LENGTH}")
print(f"Epochs:        {training_args.num_train_epochs}")
print(f"LR:            {training_args.learning_rate}")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=(
        torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else (torch.float16 if torch.cuda.is_available() else torch.float32)
    ),
    device_map="auto" if torch.cuda.is_available() else None,
)
model.config.use_cache = False

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print("SFT Trainer ready.")

In [ ]:
if RUN_TRAINING:
    result = trainer.train()
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    print(result)
    print(f"\nAdapter saved: {OUTPUT_DIR}")
else:
    print("Training skipped. Set RUN_TRAINING = True to fine-tune.")

## 7. Step 5 — Compare Baseline vs. Fine-Tuned

### The Comparison Contract

Both models are evaluated on the **same eval set** with the **same evaluation function**. The only variable that changes is the model weights.

This is the payoff for defining evals first: we get a clean, fair comparison.


In [ ]:
if RUN_FT_EVAL:
    from peft import PeftModel
    from transformers import pipeline as hf_pipeline

    ft_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    ft_model = PeftModel.from_pretrained(ft_base, str(OUTPUT_DIR))
    ft_gen = hf_pipeline("text-generation", model=ft_model, tokenizer=tokenizer)

    ft_outputs = []
    for _, row in eval_df.iterrows():
        messages = build_classify_messages(row["text"])
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        result = ft_gen(prompt, max_new_tokens=20, do_sample=False)
        output = result[0]["generated_text"][len(prompt):].strip()
        ft_outputs.append(output)

    ft_report, ft_preds = evaluate(
        eval_df["label"].tolist(), ft_outputs, run_name="fine_tuned"
    )
    print_report(ft_report)
    plot_confusion(eval_df["label"].tolist(), ft_preds, title="Fine-Tuned")
else:
    # Simulate fine-tuned results: ~85% accuracy (significant improvement)
    rng_ft = np.random.default_rng(SEED + 100)
    sim_ft_outputs = []
    for _, row in eval_df.iterrows():
        true = row["label"]
        if rng_ft.random() < 0.85:
            sim_ft_outputs.append(true)
        else:
            neighbors = {
                "P0-critical": ["P1-high"],
                "P1-high": ["P2-medium"],
                "P2-medium": ["P1-high", "P3-low"],
                "P3-low": ["P2-medium"],
            }
            sim_ft_outputs.append(rng_ft.choice(neighbors[true]))

    ft_report, ft_preds = evaluate(
        eval_df["label"].tolist(), sim_ft_outputs, run_name="fine_tuned"
    )
    print_report(ft_report)
    print("(Simulated results. Set RUN_FT_EVAL = True for real evaluation.)")

In [ ]:
plot_confusion(eval_df["label"].tolist(), ft_preds, title="Fine-Tuned")

In [ ]:
# Side-by-side comparison
comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Macro F1", "Parse Rate", "Mean Severity Dist.", "Severe Error Rate"],
    "Baseline": [
        baseline_report["accuracy"],
        baseline_report["macro_f1"],
        baseline_report["parse_rate"],
        baseline_report["mean_severity_distance"],
        baseline_report["severe_error_rate"],
    ],
    "Fine-Tuned": [
        ft_report["accuracy"],
        ft_report["macro_f1"],
        ft_report["parse_rate"],
        ft_report["mean_severity_distance"],
        ft_report["severe_error_rate"],
    ],
})
comparison["Delta"] = comparison["Fine-Tuned"] - comparison["Baseline"]
comparison["Direction"] = comparison["Delta"].apply(
    lambda d: "better" if (d > 0 and "Severity" not in "") else ("better" if d < 0 else "same")
)
# Fix direction for distance & error metrics (lower is better)
for idx in comparison.index:
    metric = comparison.loc[idx, "Metric"]
    if "Severity" in metric or "Error" in metric:
        comparison.loc[idx, "Direction"] = "better" if comparison.loc[idx, "Delta"] < 0 else "worse" if comparison.loc[idx, "Delta"] > 0 else "same"

print("SIDE-BY-SIDE COMPARISON")
print("=" * 70)
print(comparison.to_string(index=False))

In [ ]:
# Visual comparison: per-class F1
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(CLASSES))
width = 0.35

baseline_f1s = [baseline_report["per_class"][c]["f1-score"] for c in CLASSES]
ft_f1s = [ft_report["per_class"][c]["f1-score"] for c in CLASSES]

bars1 = ax.bar(x - width / 2, baseline_f1s, width, label="Baseline Prompt", color="#d62728", alpha=0.8)
bars2 = ax.bar(x + width / 2, ft_f1s, width, label="Fine-Tuned", color="#2ca02c", alpha=0.8)

ax.set_xlabel("Priority Class")
ax.set_ylabel("F1 Score")
ax.set_title("Per-Class F1: Baseline vs. Fine-Tuned")
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, fontsize=9)
ax.set_ylim([0, 1.1])
ax.legend()
ax.axhline(0.9, color="gray", linestyle="--", linewidth=0.7, alpha=0.6)

for bar, val in zip(list(bars1) + list(bars2), baseline_f1s + ft_f1s):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.0%}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

## 8. Step 6 — Error Analysis

### What to Look For

Numbers tell you **how much** improved. Error analysis tells you **what** improved and **what still fails**.

We answer three diagnostic questions:

1. **Which errors did fine-tuning fix?** (baseline wrong → FT correct)
2. **Which errors did fine-tuning introduce?** (baseline correct → FT wrong) — regressions
3. **Which errors persist in both?** (both wrong) — these need more training data or better labels


In [ ]:
error_analysis = pd.DataFrame({
    "text": eval_df["text"],
    "true": eval_df["label"],
    "baseline_pred": baseline_preds,
    "ft_pred": ft_preds,
})

error_analysis["baseline_correct"] = error_analysis["true"] == error_analysis["baseline_pred"]
error_analysis["ft_correct"] = error_analysis["true"] == error_analysis["ft_pred"]

# Categorize each example
def categorize(row):
    if row["baseline_correct"] and row["ft_correct"]:
        return "both_correct"
    elif not row["baseline_correct"] and row["ft_correct"]:
        return "ft_fixed"
    elif row["baseline_correct"] and not row["ft_correct"]:
        return "ft_regression"
    else:
        return "both_wrong"

error_analysis["category"] = error_analysis.apply(categorize, axis=1)

cat_counts = error_analysis["category"].value_counts()
print("ERROR ANALYSIS CATEGORIES")
print("=" * 50)
for cat in ["both_correct", "ft_fixed", "ft_regression", "both_wrong"]:
    count = cat_counts.get(cat, 0)
    pct = count / len(error_analysis) * 100
    icon = {"both_correct": "ok", "ft_fixed": "IMPROVED", "ft_regression": "REGRESSED", "both_wrong": "STILL WRONG"}[cat]
    print(f"  {icon:<12} {cat:<16} {count:>3} ({pct:.0f}%)")

In [ ]:
# Show the tickets fine-tuning fixed
fixed = error_analysis[error_analysis["category"] == "ft_fixed"]
if len(fixed):
    print(f"TICKETS FINE-TUNING FIXED ({len(fixed)}):")
    print("-" * 70)
    for _, row in fixed.iterrows():
        print(f"  True: {row['true']:<14}  Baseline: {row['baseline_pred']:<14}  FT: {row['ft_pred']}")
        print(f"  Text: {row['text'][:75]}...")
        print()

# Show regressions
regressed = error_analysis[error_analysis["category"] == "ft_regression"]
if len(regressed):
    print(f"\nREGRESSIONS ({len(regressed)}):")
    print("-" * 70)
    for _, row in regressed.iterrows():
        print(f"  True: {row['true']:<14}  Baseline: {row['baseline_pred']:<14}  FT: {row['ft_pred']}")
        print(f"  Text: {row['text'][:75]}...")
        print()

# Show persistent errors
both_wrong = error_analysis[error_analysis["category"] == "both_wrong"]
if len(both_wrong):
    print(f"\nPERSISTENT ERRORS ({len(both_wrong)}):")
    print("-" * 70)
    for _, row in both_wrong.iterrows():
        print(f"  True: {row['true']:<14}  Baseline: {row['baseline_pred']:<14}  FT: {row['ft_pred']}")
        print(f"  Text: {row['text'][:75]}...")
        print()

In [ ]:
# Severity analysis: did fine-tuning reduce dangerous misclassifications?
baseline_distances = [
    severity_distance(t, p) for t, p in zip(error_analysis["true"], error_analysis["baseline_pred"])
    if p in CLASSES
]
ft_distances = [
    severity_distance(t, p) for t, p in zip(error_analysis["true"], error_analysis["ft_pred"])
    if p in CLASSES
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, dists, title, color in [
    (axes[0], baseline_distances, "Baseline Prompt", "#d62728"),
    (axes[1], ft_distances, "Fine-Tuned", "#2ca02c"),
]:
    counts = Counter(dists)
    labels_x = sorted(counts.keys())
    values = [counts[k] for k in labels_x]
    ax.bar(labels_x, values, color=color, alpha=0.8)
    ax.set_xlabel("Severity Distance (levels apart)")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.set_xticks([0, 1, 2, 3])

plt.suptitle("Distribution of Prediction Severity Distance", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Baseline — mean severity distance: {np.mean(baseline_distances):.3f}")
print(f"Fine-tuned — mean severity distance: {np.mean(ft_distances):.3f}")

## 9. Step 7 — Iterate

### The Iteration Playbook

Error analysis tells you what to do next. Here is the decision tree:

```text
Persistent errors (both wrong)?
  └─ Look at the tickets. Are the labels correct?
      └─ NO  → Fix labels in the dataset, retrain.
      └─ YES → Add more examples of this specific pattern to training data.

Regressions (baseline right, FT wrong)?
  └─ Is the regression consistent (same class pairings)?
      └─ YES → The training data may have mislabelled examples for that class.
               Audit training data for that class.
      └─ NO  → May be overfitting noise. Reduce epochs or increase training data.

FT improved but still below target?
  └─ More training data for the weakest classes.
  └─ Increase LoRA rank or unfreeze additional layers.
  └─ Add few-shot examples to the system prompt AND fine-tune.
```

### What "Done" Looks Like

Define acceptance criteria **before** iterating:

| Metric | Target |
|---|---|
| Overall accuracy | ≥ 85% |
| Macro F1 | ≥ 0.82 |
| Severe error rate (≥ 2 levels apart) | ≤ 5% |
| Parse success rate | 100% |
| No P0 classified as P3 or vice versa | 0 occurrences |


In [ ]:
# Check acceptance criteria
criteria = [
    ("Accuracy >= 85%", ft_report["accuracy"] >= 0.85),
    ("Macro F1 >= 0.82", ft_report["macro_f1"] >= 0.82),
    ("Severe error rate <= 5%", ft_report["severe_error_rate"] <= 0.05),
    ("Parse rate = 100%", ft_report["parse_rate"] >= 1.0),
]

print("ACCEPTANCE CRITERIA CHECK")
print("=" * 50)
all_pass = True
for name, passed in criteria:
    status = "PASS" if passed else "FAIL"
    icon = "pass" if passed else "FAIL"
    print(f"  [{icon}] {name}")
    if not passed:
        all_pass = False

print(f"\nOverall: {'All criteria met — ready for deployment.' if all_pass else 'Some criteria not met. Iterate.'}")

## 10. When Fine-Tuning Is Worth It

### The Decision Framework

| Baseline accuracy | Gap to target | Fine-tuning verdict |
|---|---|---|
| ≥ 95% | Small | Probably not worth it. Better prompting or few-shot may close the gap. |
| 75 – 94% | Medium | Best case for fine-tuning. Clear room for improvement. |
| < 75% | Large | Fine-tuning helps but may not be enough alone. Check data quality first. |

### Costs to Consider

| Cost | Details |
|---|---|
| **Data curation** | Labelling 40–500 examples: hours of expert time |
| **Compute** | LoRA on a 3B model: ~30 min on a single GPU |
| **Iteration** | 2–5 rounds of error analysis and retraining |
| **Maintenance** | Re-evaluate when the domain changes |

### When Prompting Is Enough

Fine-tuning is overhead. If you can reach your target accuracy by:
- improving the system prompt
- adding few-shot examples
- better output parsing

...then do that instead. Fine-tuning is for the gap that prompting cannot close.


## 11. The Full Loop — Summary

```text
┌─────────────────────────────────────────────────────────────────┐
│ STEP 1: Define evals                                            │
│   • accuracy, macro F1, per-class P/R/F1, severity distance     │
│   • parse success rate                                          │
│   • fixed BEFORE any model run                                  │
├─────────────────────────────────────────────────────────────────┤
│ STEP 2: Baseline prompting                                      │
│   • zero-shot classification with the same eval set             │
│   • measures what prompting alone achieves                      │
├─────────────────────────────────────────────────────────────────┤
│ STEP 3: Build SFT dataset                                       │
│   • same prompt template as baseline                            │
│   • encodes organization-specific priority rules                │
├─────────────────────────────────────────────────────────────────┤
│ STEP 4: Fine-tune with LoRA                                     │
│   • small adapter, few epochs, low LR                           │
│   • behavioral adjustment, not knowledge injection              │
├─────────────────────────────────────────────────────────────────┤
│ STEP 5: Compare                                                 │
│   • same eval set, same eval function                           │
│   • side-by-side metrics table                                  │
├─────────────────────────────────────────────────────────────────┤
│ STEP 6: Error analysis                                          │
│   • what was fixed, what regressed, what persists               │
│   • severity distribution shift                                 │
├─────────────────────────────────────────────────────────────────┤
│ STEP 7: Iterate                                                 │
│   • fix labels, add examples, adjust hyperparameters            │
│   • re-run from Step 4 until acceptance criteria are met        │
└─────────────────────────────────────────────────────────────────┘
```

### Key Takeaways

1. **Evals first.** Without a fixed evaluation contract, you cannot measure progress.
2. **Baseline before training.** Know what prompting achieves before investing in data.
3. **Same eval for both.** The only variable that changes is the model weights.
4. **Error analysis drives iteration.** Numbers say how much; examples say why.
5. **Define "done" before iterating.** Acceptance criteria prevent infinite loops.
6. **Fine-tuning is for the gap prompting cannot close.** If prompting gets you there, ship it.


## 12. Save Experiment Log

In [ ]:
log = {
    "timestamp": datetime.now().isoformat(),
    "task": "end_to_end_fine_tuning_loop",
    "base_model": BASE_MODEL,
    "classes": CLASSES,
    "dataset_size": len(df),
    "train_size": len(train_df),
    "eval_size": len(eval_df),
    "baseline_report": baseline_report,
    "ft_report": ft_report,
}

log_path = ARTIFACT_DIR / "e2e_finetune_loop_log.json"
log_path.write_text(json.dumps(log, indent=2, default=str), encoding="utf-8")
print(f"Saved: {log_path}")